In [ ]:
import numpy as np
from scipy.stats import multivariate_t
from limvam.praline import praline
import lingam
import warnings
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# generate data

In [ ]:
# parameters
m = 8
p = 6
n = 1000
random_state = 18
rng = np.random.RandomState(random_state)
steps = 1000
lr = 1e-2
disturbances = "student_t"

In [ ]:
def generate_data(
    m,
    p,
    n,
    rng=None,
    disturbances="gaussian",
):
    # variance of the disturbances
    M = rng.randn(p, m, m)
    Sigmas = np.zeros((p, m, m))
    for j in range(p):
        Sigmas[j] = M[j] @ M[j].T + m * np.eye(m)
    
    # disturbances
    E = np.zeros((m, p, n))
    for j in range(p):
        if disturbances == "gaussian":
            E[:, j] = rng.multivariate_normal(mean=np.zeros(m), cov=Sigmas[j], size=(n,)).T
        elif disturbances == "student_t":
            E[:, j] = multivariate_t.rvs(loc=np.zeros(m), shape=Sigmas[j], df=1, size=n, random_state=rng).T
        else:
            raise ValueError("The parameter 'disturbances' should be 'gaussian' or 'student_t'.")

    # strictly lower triangular matrices T
    T = rng.normal(size=(m, p, p))
    for i in range(m):
        T[i][np.triu_indices(p, k=0)] = 0
    
    # causal order P
    order = rng.permutation(p)
    P = np.eye(p)[order]

    # causal effect matrices B
    B = P.T @ T @ P
    
    # mixing matrices
    A = np.linalg.inv(np.eye(p) - B)
    
    # observations
    X = np.array([Ai @ Ei for Ai, Ei in zip(A, E)])
    return X, B, T, P, order

In [ ]:
X, B, T, P, order = generate_data(m, p, n, rng)

# PRaLiNE

In [ ]:
B_estimate, T_estimate, P_estimate = praline(X, steps=steps, lr=lr)
order_estimate = np.argmax(P_estimate, axis=1)

In [ ]:
print(f"True order : {order}")
print(f"Estimated order (ours) : {order_estimate}")

# MultiGroupDirectLiNGAM

In [ ]:
model = lingam.MultiGroupDirectLiNGAM()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    model.fit(list(np.swapaxes(X, 1, 2)))
order_estimate_2 = model.causal_order_

In [ ]:
print(f"True order : {order}")
print(f"Estimated order (not ours): {order_estimate_2}")

# plot

In [ ]:
# matplotlib style
fontsize = 22
rc = {
    "font.size": fontsize,
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "font.family": "serif",
}
plt.rcParams.update(rc)

In [ ]:
# plot matrices Ti
def plot_heat_maps(T, suptitle=""):
    m = len(T)
    fig, axes = plt.subplots(1, m, figsize=(m*5, 5))
    norm = TwoSlopeNorm(vmin=min(-1, np.min(T)), vmax=max(1, np.max(T)), vcenter=0)
    for i, ax in enumerate(axes.flat):
        im = ax.imshow(T[i], norm=norm, cmap="coolwarm")
        ax.set_title(f"View {i}")
    cbar = fig.colorbar(im, ax=axes, fraction=0.022, pad=0.03)
    cbar.set_label("Color Scale", fontsize=fontsize+2)
    fig.supxlabel("Variables")
    fig.supylabel("Variables", x=0.09)
    fig.suptitle(suptitle)
    plt.show()

In [ ]:
plot_heat_maps(T, suptitle=r"True matrices $T^i$")
plot_heat_maps(T_estimate, suptitle=r"Estimated matrices $T^i$")

In [ ]:
plot_heat_maps(B, suptitle=r"True matrices $B^i$")
plot_heat_maps(B_estimate, suptitle=r"Estimated matrices $B^i$")